# Chương 6b. Nhận dạng chữ số viết tay – So sánh nhiều mô hình
*Giáo trình **Học máy – Cơ sở và Ứng dụng với Python*** · Notebook thực hành mở rộng của Chương 6

Chương 6 đã học phân loại **nhị phân** bằng hồi quy Logistic. Ở đây ta mở rộng sang phân loại **đa lớp** (10 chữ số 0–9) và đặt hồi quy Logistic cạnh nhiều mô hình khác để so sánh một cách bài bản: chỉ số, thời gian, độ ổn định và khả năng chịu nhiễu.

Chạy lần lượt từng ô (**Runtime → Run all**, mất khoảng 1–2 phút). Không cần GPU, không cần tải dữ liệu ngoài.

## Mục tiêu học tập
1. Mở rộng hồi quy Logistic sang đa lớp (softmax) và đọc ma trận nhầm lẫn 10×10.
2. Phân biệt và diễn giải **Accuracy, Precision, Recall, F1 (macro/weighted), ROC-AUC (OvR), Log Loss**.
3. Phân tích lỗi: lớp nào, cặp chữ số nào hay bị nhầm, nhìn trực tiếp ảnh bị sai.
4. So sánh 7 mô hình bằng **kiểm định chéo (cross-validation)** thay vì chỉ một lần chia.
5. Rút ra kết luận có căn cứ, biết khi nào chênh lệch là **có ý nghĩa** và khi nào chỉ là nhiễu.

## 1. Dữ liệu
Dùng `load_digits` của scikit-learn: **1797 ảnh xám 8×8** (64 đặc trưng, mỗi pixel 0–16), nhãn 0–9. Đây là bản thu nhỏ cùng tinh thần với MNIST (28×28), tải sẵn nên chạy offline.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, log_loss, f1_score)

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = False
np.random.seed(42)

du_lieu = load_digits()
X_tho, y = du_lieu.data, du_lieu.target
print("Kích thước X:", X_tho.shape, "| Giá trị pixel:", X_tho.min(), "→", X_tho.max())
print("Số mẫu mỗi chữ số:", np.bincount(y))

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(du_lieu.images[i], cmap="gray_r"); ax.set_title(f"nhãn {y[i]}", fontsize=9); ax.axis("off")
plt.suptitle("20 ảnh đầu tiên (8×8 pixel)"); plt.tight_layout(); plt.show()

# Ảnh "trung bình" của mỗi chữ số
fig, axes = plt.subplots(1, 10, figsize=(13, 1.9))
for k, ax in enumerate(axes):
    ax.imshow(du_lieu.images[y == k].mean(axis=0), cmap="gray_r"); ax.set_title(str(k)); ax.axis("off")
plt.suptitle("Ảnh trung bình của từng chữ số"); plt.tight_layout(); plt.show()

**Nhận xét.** Các lớp khá cân bằng (~180 ảnh mỗi chữ số) nên **Accuracy là thước đo hợp lý**, nhưng ta vẫn xem thêm Precision/Recall/F1 từng lớp. Ảnh trung bình cho thấy những chữ số có hình dạng gần nhau (ví dụ 3–5–8, 4–9, 1–7) là ứng viên bị nhầm.

## 2. Tiền xử lý
- Chia 75% huấn luyện / 25% kiểm tra, `stratify=y` để giữ tỉ lệ các chữ số.
- Co pixel về $[0,1]$ bằng cách chia 16. Với các mô hình nhạy với thang đo (Logistic, SVM, MLP) ta thêm `StandardScaler` trong **pipeline** để chuẩn hoá chỉ dựa trên tập huấn luyện (tránh rò rỉ dữ liệu).

In [ ]:
X = X_tho / 16.0
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("Huấn luyện:", X_train.shape, "| Kiểm tra:", X_test.shape)

## 3. Mô hình cơ sở: hồi quy Logistic đa lớp (softmax)
Với $K=10$ lớp, mô hình có một vector trọng số $\mathbf w_k$ cho mỗi lớp, tính điểm $z_k=\mathbf w_k^\top\mathbf x+b_k$ rồi dùng **softmax**:
$$P(y=k\mid\mathbf x)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}$$
Hàm mất mát là Cross-Entropy tổng quát $L=-\frac1m\sum_i\log P(y=y_i\mid\mathbf x_i)$ (mở rộng của Log Loss nhị phân ở Chương 6). Dự đoán là lớp có xác suất lớn nhất.

In [ ]:
mo_hinh_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
mo_hinh_lr.fit(X_train, y_train)
y_dd = mo_hinh_lr.predict(X_test)

print(f"Accuracy = {accuracy_score(y_test, y_dd):.4f}\n")
print(classification_report(y_test, y_dd, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_dd, ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Ma trận nhầm lẫn – Logistic Regression")
plt.show()

### Cách đọc báo cáo phân loại đa lớp
- **Precision của lớp $k$**: trong các ảnh *dự đoán là $k$*, bao nhiêu % đúng. Thấp ⇒ nhiều ảnh khác bị "hút" nhầm vào lớp $k$.
- **Recall của lớp $k$**: trong các ảnh *thật sự là $k$*, bao nhiêu % được tìm ra. Thấp ⇒ chữ số $k$ hay bị nhận thành chữ số khác.
- **F1** là trung bình điều hoà của hai độ đo trên.
- **macro avg**: trung bình *không trọng số* của các lớp (mọi lớp quan trọng như nhau). **weighted avg**: trung bình có trọng số theo số mẫu mỗi lớp. Khi lớp cân bằng, hai số này gần nhau.
- **Accuracy** = tổng đường chéo / tổng số mẫu.

## 4. Phân tích lỗi
Xem những cặp chữ số bị nhầm nhiều nhất và trực tiếp nhìn các ảnh bị đoán sai.

In [ ]:
cm = confusion_matrix(y_test, y_dd)
loi = cm.copy(); np.fill_diagonal(loi, 0)
cap = [(int(i), int(j), int(loi[i, j])) for i in range(10) for j in range(10) if loi[i, j] > 0]
cap.sort(key=lambda t: -t[2])
print("Các cặp (thật → đoán) bị nhầm nhiều nhất:")
for t, d_, n in cap[:6]:
    print(f"  {t} → {d_}: {n} ảnh")

sai = np.where(y_dd != y_test)[0]
print(f"\nTổng số ảnh sai: {len(sai)} / {len(y_test)}")
n_hien = min(len(sai), 12)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for ax, idx in zip(axes.ravel(), sai[:n_hien]):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"thật {y_test[idx]} → đoán {y_dd[idx]}", fontsize=9, color="crimson"); ax.axis("off")
for ax in axes.ravel()[n_hien:]: ax.axis("off")
plt.suptitle("Các ảnh bị phân loại sai"); plt.tight_layout(); plt.show()

**Nhận xét.** Các ảnh sai thường là chữ viết *méo, đứt nét hoặc mờ* — ở độ phân giải 8×8, ngay cả mắt người cũng khó phân biệt một số ảnh. Điều này giải thích vì sao Accuracy khó đạt 100% trên bộ dữ liệu nhỏ này.

## 5. Ảnh hưởng của điều chuẩn $C$ trong Logistic Regression
Trong scikit-learn, $C=1/\lambda$ là nghịch đảo cường độ điều chuẩn L2: $C$ nhỏ ⇒ phạt trọng số mạnh (mô hình đơn giản, dễ **underfit**); $C$ lớn ⇒ ít điều chuẩn (dễ **overfit**).

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Cs = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
tb_tr, tb_cv = [], []
for C in Cs:
    mh = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))
    mh.fit(X_train, y_train)
    tb_tr.append(mh.score(X_train, y_train))
    tb_cv.append(cross_val_score(mh, X_train, y_train, cv=skf).mean())

plt.semilogx(Cs, tb_tr, "o-", label="Huấn luyện")
plt.semilogx(Cs, tb_cv, "s-", label="Kiểm định chéo (5-fold)")
plt.xlabel("C (điều chuẩn yếu →)"); plt.ylabel("Accuracy"); plt.grid(alpha=0.3); plt.legend()
plt.title("Chọn C cho Logistic Regression"); plt.show()
C_tot = Cs[int(np.argmax(tb_cv))]
print("C tốt nhất theo CV:", C_tot, "| CV accuracy =", round(max(tb_cv), 4))

**Nhận xét.** Với $C$ rất nhỏ, mô hình bị điều chuẩn quá mức nên cả điểm huấn luyện lẫn CV đều thấp (underfit). Khi $C$ tăng, điểm huấn luyện tiến dần tới 1,0 trong khi điểm CV đạt đỉnh rồi đi ngang/giảm nhẹ — khoảng cách giữa hai đường là dấu hiệu overfit. Ta chọn $C$ theo **điểm CV**, không phải điểm huấn luyện.

## 6. So sánh nhiều mô hình
Bảy mô hình đại diện cho các họ thuật toán khác nhau:

| Mô hình | Ý tưởng |
|---|---|
| Logistic Regression | tuyến tính + softmax |
| K-NN (k=3) | nhìn các ảnh huấn luyện gần nhất |
| SVM (RBF) | lề cực đại với ranh giới phi tuyến |
| Decision Tree | chuỗi câu hỏi if–else trên pixel |
| Random Forest | 200 cây, bỏ phiếu |
| Naive Bayes (Gauss) | giả định pixel độc lập có điều kiện |
| MLP (64 nơ-ron ẩn) | mạng nơ-ron nông |

**Quy trình công bằng:** cùng tập huấn luyện/kiểm tra; ước lượng bằng **5-fold CV trên tập huấn luyện** (độ ổn định), sau đó chấm **một lần** trên tập kiểm tra; ghi lại thời gian.

In [ ]:
mo_hinh = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "K-NN (k=3)":          KNeighborsClassifier(n_neighbors=3),
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", probability=True, random_state=42)),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Naive Bayes":         GaussianNB(),
    "MLP":                 make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)),
}

dong, cv_diem, du_doan, xac_suat = [], {}, {}, {}
for ten, mh in mo_hinh.items():
    diem = cross_val_score(mh, X_train, y_train, cv=skf)
    cv_diem[ten] = diem
    t0 = time.perf_counter(); mh.fit(X_train, y_train); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); p = mh.predict(X_test);   t_pred = time.perf_counter() - t0
    pr = mh.predict_proba(X_test)
    du_doan[ten], xac_suat[ten] = p, pr
    P, R, F1, _ = precision_recall_fscore_support(y_test, p, average="macro")
    dong.append({"Mô hình": ten,
                 "CV mean": diem.mean(), "CV std": diem.std(),
                 "Accuracy": accuracy_score(y_test, p),
                 "Precision (macro)": P, "Recall (macro)": R, "F1 (macro)": F1,
                 "ROC-AUC (OvR)": roc_auc_score(y_test, pr, multi_class="ovr"),
                 "Log Loss": log_loss(y_test, pr, labels=range(10)),
                 "Huấn luyện (s)": t_fit, "Dự đoán (ms)": t_pred * 1000})

bang = pd.DataFrame(dong).set_index("Mô hình")
bang_hien_thi = bang.sort_values("Accuracy", ascending=False).round(4)
bang_hien_thi

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
thu_tu = bang.sort_values("Accuracy").index

ax[0].barh(thu_tu, bang.loc[thu_tu, "Accuracy"], color="steelblue")
ax[0].set_xlim(0.75, 1.0); ax[0].set_title("Accuracy trên tập kiểm tra")
for i, v in enumerate(bang.loc[thu_tu, "Accuracy"]): ax[0].text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8)

ax[1].boxplot([cv_diem[m] for m in thu_tu], vert=False)
ax[1].set_yticklabels(thu_tu)
ax[1].set_title("Accuracy qua 5-fold CV (độ ổn định)"); ax[1].set_xlim(0.75, 1.0)

ax[2].scatter(bang["Huấn luyện (s)"], bang["Accuracy"], s=80, c="crimson")
for m, r in bang.iterrows(): ax[2].annotate(m, (r["Huấn luyện (s)"], r["Accuracy"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[2].set_xscale("log"); ax[2].set_xlabel("Thời gian huấn luyện (s, thang log)"); ax[2].set_ylabel("Accuracy")
ax[2].set_title("Độ chính xác so với chi phí"); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# F1 theo từng chữ số của từng mô hình
f1_lop = pd.DataFrame({m: f1_score(y_test, p, average=None) for m, p in du_doan.items()},
                      index=[str(i) for i in range(10)])
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(f1_lop.T.values, cmap="RdYlGn", vmin=0.6, vmax=1.0, aspect="auto")
ax.set_xticks(range(10)); ax.set_xticklabels(f1_lop.index); ax.set_yticks(range(len(f1_lop.columns)))
ax.set_yticklabels(f1_lop.columns); ax.set_xlabel("Chữ số")
for i in range(len(f1_lop.columns)):
    for j in range(10): ax.text(j, i, f"{f1_lop.iloc[j, i]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, label="F1"); ax.set_title("F1 theo từng chữ số và từng mô hình")
plt.tight_layout(); plt.show()

print("Chữ số khó nhất (F1 trung bình thấp nhất qua các mô hình):")
print(f1_lop.mean(axis=1).sort_values().head(3).round(3).to_string())

## 7. Kiểm tra: chênh lệch giữa các mô hình có ý nghĩa không?
Tập kiểm tra chỉ có 450 ảnh, nên 1 ảnh sai chênh ~0,22% Accuracy. Ta so sánh khoảng dao động CV: nếu hai mô hình có `mean ± std` chồng lấn nhiều thì chênh lệch rất có thể chỉ là ngẫu nhiên.

In [ ]:
xep = bang.sort_values("CV mean", ascending=False)
tot_nhat = xep.index[0]
print(f"Mô hình CV tốt nhất: {tot_nhat} ({xep.iloc[0]['CV mean']:.4f} ± {xep.iloc[0]['CV std']:.4f})\n")
for m, r in xep.iloc[1:].iterrows():
    chenh = xep.iloc[0]["CV mean"] - r["CV mean"]
    nguong = xep.iloc[0]["CV std"] + r["CV std"]
    ket_luan = "không phân biệt được (trong ngưỡng nhiễu)" if chenh <= nguong else "kém hơn rõ rệt"
    print(f"  {m:22s} chênh {chenh:+.4f} so với ngưỡng nhiễu {nguong:.4f} → {ket_luan}")

In [ ]:
# Ma trận nhầm lẫn của mô hình tốt nhất trên tập kiểm tra
tot_test = bang["Accuracy"].idxmax()
fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(y_test, du_doan[tot_test], ax=ax, cmap="Greens", colorbar=False)
ax.set_title(f"Ma trận nhầm lẫn – {tot_test}"); plt.show()
print("Mô hình tốt nhất trên tập kiểm tra:", tot_test)

## 8. Nhận xét các chỉ số

*(Số liệu dưới đây là kết quả của lần chạy mẫu với `random_state=42`; có thể lệch nhẹ tuỳ phiên bản thư viện, nhưng thứ tự và kết luận chung ổn định. Bảng ở Mục 6 là nguồn số liệu chính thức khi bạn chạy.)*

**① Accuracy và độ ổn định.** Năm mô hình **K-NN (98,4%), SVM (98,0%), MLP (98,0%), Logistic Regression (97,8%), Random Forest (96,7%)** đều nằm trong khoảng **96,5–98,5%** trên tập kiểm tra, trong khi **Decision Tree (~82%)** và **Naive Bayes (~83%)** thấp hơn hẳn (~15 điểm %). Trong nhóm đầu, khoảng cách chỉ là vài ảnh trên 450 ảnh kiểm tra. Theo CV, K-NN dẫn đầu (~98,8%); SVM, Random Forest, MLP nằm trong ngưỡng nhiễu so với K-NN (Mục 7) nên **không thể khẳng định** ai hơn ai; riêng Logistic Regression (~96,6%) kém K-NN khoảng 2 điểm % — chênh lệch này vượt ngưỡng nhiễu. Bài học: xếp hạng trên *một* lần chia train/test có thể đảo khi đổi `random_state`; hãy dựa vào CV.

**② Precision, Recall, F1 (macro).** Vì 10 lớp gần như cân bằng, Precision, Recall và F1 macro gần như trùng với Accuracy ở các mô hình tốt — không có lớp nào bị hy sinh. Khi các số này lệch nhau mới là dấu hiệu mô hình thiên vị một số lớp: Naive Bayes có Precision macro (~0,86) cao hơn Recall macro (~0,83) rõ rệt, nghĩa là với một số chữ số nó rất "dè dặt" khi gán nhãn nên bỏ sót nhiều ảnh thật của lớp đó (xem Recall từng lớp để biết lớp nào).

**③ ROC-AUC (OvR) và Log Loss.** AUC của năm mô hình tốt đều **≥ 0,99** nên gần như không phân biệt được chúng (AUC đo khả năng *xếp hạng* và bão hoà sớm trên bài toán dễ); Decision Tree (~0,90) và Naive Bayes (~0,96) thấp hơn. **Log Loss** phân biệt rõ hơn vì phạt các dự đoán *sai mà tự tin*: Logistic Regression (~0,10) và MLP (~0,10) tốt nhất, SVM (~0,12) gần đó; Random Forest (~0,33) và **K-NN (~0,49)** cao hơn nhiều — dù K-NN có Accuracy cao nhất, xác suất của nó thô (chỉ nhận các giá trị 0, 1/3, 2/3, 1 với k=3) nên bị phạt nặng khi sai. Decision Tree (~6,3) và Naive Bayes (~3,5) cực cao vì đưa ra xác suất gần 0/1 một cách quá tự tin. Khi cần *xác suất đáng tin* (không chỉ nhãn), hãy nhìn Log Loss hoặc **hiệu chỉnh xác suất**, đừng chỉ nhìn Accuracy.

**④ Phân tích theo lớp.** Biểu đồ nhiệt F1 cho thấy chữ số **8 là khó nhất ở hầu hết mô hình** (F1 trung bình thấp nhất), tiếp theo là **9 và 1**; với Logistic Regression, cặp nhầm nhiều nhất là 8→1 (3 ảnh), rồi 1→8. Các mô hình mạnh chỉ giảm nhẹ ở những chữ số này (F1 ~0,92–0,97), còn Decision Tree và Naive Bayes sụt mạnh ở nhiều chữ số cùng lúc (F1 của chữ số 8 chỉ ~0,65–0,68).

**⑤ Chi phí.** Decision Tree, Naive Bayes và K-NN huấn luyện gần như tức thì (K-NN chỉ lưu dữ liệu, nhưng phải so với mọi ảnh huấn luyện lúc dự đoán — sẽ chậm khi dữ liệu lớn). Trong lần chạy mẫu, Random Forest tốn nhiều thời gian nhất cả lúc huấn luyện lẫn dự đoán, SVM dự đoán chậm hơn Logistic/MLP khoảng hai chục lần. Logistic Regression vừa nhanh, vừa chính xác, vừa **dễ giải thích**, lại có Log Loss thấp nhất nhóm — một baseline rất đáng giá. (Thời gian đo được dao động tuỳ máy; hãy xem thứ tự lớn hơn là con số tuyệt đối.)

**⑥ Kết luận thực tế.** Trên bài toán này, mô hình đơn giản (Logistic Regression) chỉ kém nhóm đầu khoảng 0,5–2 điểm %, trong khi lại cho xác suất tốt nhất và nhanh; chênh lệch giữa các mô hình mạnh phần lớn nhỏ hơn hoặc ngang nhiễu thống kê. Nên chọn theo *chi phí – khả năng giải thích – nhu cầu xác suất – độ bền vững (Bài 3)* thay vì chạy theo 0,2% Accuracy. Lưu ý đây là ảnh 8×8 sạch; trên MNIST 28×28 hoặc chữ viết thật, khoảng cách giữa mô hình tuyến tính và mạng nơ-ron tích chập (CNN) sẽ lớn hơn nhiều.

## 9. Bài tập kèm lời giải đầy đủ

### Bài 1 – Vai trò của chuẩn hoá
**Đề.** Huấn luyện Logistic Regression, SVM (RBF) và K-NN **có** và **không** có `StandardScaler`. Mô hình nào chịu ảnh hưởng nhiều nhất, theo hướng nào? Giải thích.

<details>
<summary>✅ Lời giải Bài 1</summary>

```python
rows = []
cau_hinh = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=2000),
    "SVM (RBF)":           lambda: SVC(C=10, gamma="scale"),
    "K-NN (k=3)":          lambda: KNeighborsClassifier(3),
}
for ten, tao in cau_hinh.items():
    kq = {"Mô hình": ten}
    for nhan, mh in {"Có chuẩn hoá": make_pipeline(StandardScaler(), tao()), "Không chuẩn hoá": tao()}.items():
        mh.fit(X_train, y_train)
        kq[nhan] = accuracy_score(y_test, mh.predict(X_test))
    kq["Chênh lệch"] = kq["Có chuẩn hoá"] - kq["Không chuẩn hoá"]
    rows.append(kq)
bai1 = pd.DataFrame(rows).set_index("Mô hình").round(4)
bai1
```

**Nhận xét (kết quả mẫu):** chuẩn hoá giúp Logistic Regression (~97,8% so với ~96,2%) nhưng lại **làm giảm** SVM (~98,0% so với ~99,1%) và K-NN (~96,7% so với ~98,4%). Lý do: pixel đã nằm sẵn trong thang $[0,1]$, còn `StandardScaler` chia cho độ lệch chuẩn nên *khuếch đại các pixel rìa gần như luôn bằng 0* (phương sai rất nhỏ, ít thông tin) lên ngang với pixel quan trọng. Kết luận: không phải lúc nào chuẩn hoá cũng tốt — phải đo bằng dữ liệu. (Lưu ý: trong Mục 6 ta vẫn dùng pipeline có `StandardScaler` cho SVM/MLP/Logistic; xem Bài 3 để thấy hệ quả.)

</details>

In [ ]:
rows = []
cau_hinh = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=2000),
    "SVM (RBF)":           lambda: SVC(C=10, gamma="scale"),
    "K-NN (k=3)":          lambda: KNeighborsClassifier(3),
}
for ten, tao in cau_hinh.items():
    kq = {"Mô hình": ten}
    for nhan, mh in {"Có chuẩn hoá": make_pipeline(StandardScaler(), tao()), "Không chuẩn hoá": tao()}.items():
        mh.fit(X_train, y_train)
        kq[nhan] = accuracy_score(y_test, mh.predict(X_test))
    kq["Chênh lệch"] = kq["Có chuẩn hoá"] - kq["Không chuẩn hoá"]
    rows.append(kq)
bai1 = pd.DataFrame(rows).set_index("Mô hình").round(4)
bai1

### Bài 2 – Chọn $k$ cho K-NN
**Đề.** Quét $k=1,\dots,15$, vẽ Accuracy CV theo $k$ và chọn $k$ tốt nhất. Giải thích hiện tượng khi $k$ quá nhỏ và quá lớn.

<details>
<summary>✅ Lời giải Bài 2</summary>

```python
ks = range(1, 16)
cv_k = [cross_val_score(KNeighborsClassifier(k), X_train, y_train, cv=skf).mean() for k in ks]
plt.plot(list(ks), cv_k, "o-"); plt.xlabel("k"); plt.ylabel("Accuracy CV (5-fold)")
plt.title("Chọn k cho K-NN"); plt.grid(alpha=0.3); plt.show()
k_tot = list(ks)[int(np.argmax(cv_k))]
print("k tốt nhất:", k_tot, "| CV =", round(max(cv_k), 4))
```

**Nhận xét:** $k$ quá nhỏ (đặc biệt $k=1$) → ranh giới rất răng cưa, nhạy với ảnh nhiễu (phương sai cao/overfit); $k$ quá lớn → bỏ phiếu bị chi phối bởi các ảnh xa, ranh giới quá trơn (thiên lệch cao/underfit). Đường CV thường phẳng ở khoảng $k=1$–$5$ và giảm dần khi $k$ lớn.

</details>

In [ ]:
ks = range(1, 16)
cv_k = [cross_val_score(KNeighborsClassifier(k), X_train, y_train, cv=skf).mean() for k in ks]
plt.plot(list(ks), cv_k, "o-"); plt.xlabel("k"); plt.ylabel("Accuracy CV (5-fold)")
plt.title("Chọn k cho K-NN"); plt.grid(alpha=0.3); plt.show()
k_tot = list(ks)[int(np.argmax(cv_k))]
print("k tốt nhất:", k_tot, "| CV =", round(max(cv_k), 4))

### Bài 3 – Độ bền vững trước nhiễu
**Đề.** Cộng nhiễu Gauss với độ lệch chuẩn $\sigma\in\{0, 0.1, 0.2, 0.3, 0.4\}$ (trên thang $[0,1]$) vào **ảnh kiểm tra** (giữ nguyên mô hình đã huấn luyện), vẽ Accuracy theo $\sigma$ cho các mô hình. Mô hình nào bền nhất, và vì sao một số mô hình sụp đổ? Kiểm chứng nguyên nhân bằng cách thử SVM và Logistic **không** có `StandardScaler`.

<details>
<summary>✅ Lời giải Bài 3</summary>

```python
rng = np.random.default_rng(0)
sigmas = [0, 0.1, 0.2, 0.3, 0.4]
ket_qua = {m: [] for m in mo_hinh}
for s in sigmas:
    X_nhieu = np.clip(X_test + rng.normal(0, s, X_test.shape), 0, 1)
    for m, mh in mo_hinh.items():
        ket_qua[m].append(accuracy_score(y_test, mh.predict(X_nhieu)))

for m, v in ket_qua.items():
    plt.plot(sigmas, v, "o-", label=m)
plt.xlabel("σ của nhiễu"); plt.ylabel("Accuracy"); plt.title("Độ bền vững trước nhiễu")
plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.show()
bai3 = pd.DataFrame(ket_qua, index=sigmas).round(3)
bai3.index.name = "σ"
print(bai3.to_string())

# Kiểm chứng nguyên nhân: bỏ StandardScaler
X_nhieu = np.clip(X_test + np.random.default_rng(0).normal(0, 0.2, X_test.shape), 0, 1)
print("\nσ = 0.2 | Accuracy sạch → có nhiễu")
for ten, mh in {"SVM + StandardScaler": mo_hinh["SVM (RBF)"],
                "SVM không scaler": SVC(C=10).fit(X_train, y_train),
                "Logistic + StandardScaler": mo_hinh["Logistic Regression"],
                "Logistic không scaler": LogisticRegression(max_iter=2000).fit(X_train, y_train)}.items():
    print(f"  {ten:28s} {mh.score(X_test, y_test):.3f} → {mh.score(X_nhieu, y_test):.3f}")
print("Số pixel gần như hằng số (std < 0.02):", int((X_train.std(axis=0) < 0.02).sum()), "/ 64")
```

**Nhận xét (kết quả mẫu):** mọi mô hình đều giảm khi nhiễu tăng, nhưng mức độ khác nhau rất lớn. **K-NN bền nhất** (~88% ở σ=0,4), rồi **Random Forest** (~72%); Logistic Regression giảm mạnh (~30%), còn **SVM, MLP và Naive Bayes sụp xuống gần mức đoán bừa (~10–16%) ngay từ σ=0,1–0,2**. Thủ phạm chính là `StandardScaler`: một số pixel rìa gần như luôn bằng 0 trong tập huấn luyện nên độ lệch chuẩn rất nhỏ (11/64 pixel có std < 0,02); khi cộng nhiễu, các pixel này bị chia cho một số rất nhỏ và bị khuếch đại thành giá trị khổng lồ, lấn át cả ảnh. Kiểm chứng: bỏ scaler thì SVM ở σ=0,2 vẫn ~97% (so với ~10% khi có scaler) và Logistic ~95% (so với ~56%). Bài học: (1) mô hình tốt nhất trên dữ liệu sạch chưa chắc bền nhất khi dữ liệu bị nhiễu/đổi phân phối; (2) bước tiền xử lý cũng là một nguồn rủi ro — luôn kiểm tra mô hình trong điều kiện gần thực tế.

</details>

In [ ]:
rng = np.random.default_rng(0)
sigmas = [0, 0.1, 0.2, 0.3, 0.4]
ket_qua = {m: [] for m in mo_hinh}
for s in sigmas:
    X_nhieu = np.clip(X_test + rng.normal(0, s, X_test.shape), 0, 1)
    for m, mh in mo_hinh.items():
        ket_qua[m].append(accuracy_score(y_test, mh.predict(X_nhieu)))

for m, v in ket_qua.items():
    plt.plot(sigmas, v, "o-", label=m)
plt.xlabel("σ của nhiễu"); plt.ylabel("Accuracy"); plt.title("Độ bền vững trước nhiễu")
plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.show()
bai3 = pd.DataFrame(ket_qua, index=sigmas).round(3)
bai3.index.name = "σ"
print(bai3.to_string())

# Kiểm chứng nguyên nhân: bỏ StandardScaler
X_nhieu = np.clip(X_test + np.random.default_rng(0).normal(0, 0.2, X_test.shape), 0, 1)
print("\nσ = 0.2 | Accuracy sạch → có nhiễu")
for ten, mh in {"SVM + StandardScaler": mo_hinh["SVM (RBF)"],
                "SVM không scaler": SVC(C=10).fit(X_train, y_train),
                "Logistic + StandardScaler": mo_hinh["Logistic Regression"],
                "Logistic không scaler": LogisticRegression(max_iter=2000).fit(X_train, y_train)}.items():
    print(f"  {ten:28s} {mh.score(X_test, y_test):.3f} → {mh.score(X_nhieu, y_test):.3f}")
print("Số pixel gần như hằng số (std < 0.02):", int((X_train.std(axis=0) < 0.02).sum()), "/ 64")

### Bài 4 – Đọc báo cáo và rút ra khuyến nghị
**Đề.** Từ bảng `bang` (Mục 6), hãy (a) chọn mô hình triển khai cho một ứng dụng đọc mã bưu chính trên thiết bị yếu, cần dự đoán nhanh, giải thích được; (b) chỉ ra chữ số bị nhầm nhiều nhất và đề xuất cách cải thiện.

<details>
<summary>✅ Lời giải Bài 4</summary>

```python
ung_vien = bang[bang["Accuracy"] >= bang["Accuracy"].max() - 0.015]          # trong 1,5 điểm % của mô hình tốt nhất
chon = ung_vien.sort_values("Dự đoán (ms)").index[0]                          # nhanh nhất trong nhóm
print("Ứng viên (Accuracy sát mô hình tốt nhất):", list(ung_vien.index))
print("→ Chọn:", chon, "vì dự đoán nhanh nhất trong nhóm có Accuracy sát nhất.")

cm_chon = confusion_matrix(y_test, du_doan[chon]).copy()
np.fill_diagonal(cm_chon, 0)
t, d_ = np.unravel_index(cm_chon.argmax(), cm_chon.shape)
print(f"Cặp nhầm nhiều nhất của {chon}: thật {t} → đoán {d_} ({cm_chon[t, d_]} ảnh)")
```

**Khuyến nghị:** (a) trong nhóm mô hình có Accuracy sát nhau, ưu tiên mô hình **dự đoán nhanh, nhỏ gọn, dễ giải thích** — ở đây code chọn Logistic Regression (dự đoán nhanh nhất trong nhóm, ~97,8%). Nếu ảnh thực tế có thể nhiễu, hãy cân nhắc K-NN hoặc bỏ `StandardScaler` (xem Bài 3); (b) với cặp chữ số hay nhầm, thu thập thêm dữ liệu cho các chữ số đó, tăng độ phân giải ảnh, dùng tăng cường dữ liệu (xoay/dịch nhẹ) hoặc chuyển sang CNN.

</details>

In [ ]:
ung_vien = bang[bang["Accuracy"] >= bang["Accuracy"].max() - 0.015]          # trong 1,5 điểm % của mô hình tốt nhất
chon = ung_vien.sort_values("Dự đoán (ms)").index[0]                          # nhanh nhất trong nhóm
print("Ứng viên (Accuracy sát mô hình tốt nhất):", list(ung_vien.index))
print("→ Chọn:", chon, "vì dự đoán nhanh nhất trong nhóm có Accuracy sát nhất.")

cm_chon = confusion_matrix(y_test, du_doan[chon]).copy()
np.fill_diagonal(cm_chon, 0)
t, d_ = np.unravel_index(cm_chon.argmax(), cm_chon.shape)
print(f"Cặp nhầm nhiều nhất của {chon}: thật {t} → đoán {d_} ({cm_chon[t, d_]} ảnh)")

## 10. Bài tập tự luyện (không có đáp án mẫu)
1. Thêm `HistGradientBoostingClassifier` vào bảng so sánh. Nó đứng ở đâu về Accuracy, Log Loss và thời gian?
2. Đổi `random_state` của `train_test_split` qua 5 giá trị khác nhau; xếp hạng 7 mô hình có đổi không? Vẽ boxplot Accuracy theo mô hình.
3. Với SVM, quét lưới $C\in\{0.1,1,10,100\}$ và `gamma` $\in\{0.001,0.01,0.1\}$ bằng `GridSearchCV`. Cải thiện bao nhiêu so với cấu hình mặc định?
4. Dùng `calibration_curve` hoặc `CalibratedClassifierCV` để hiệu chỉnh xác suất của Random Forest và so sánh Log Loss trước/sau.
5. Tăng cường dữ liệu: dịch ảnh huấn luyện ±1 pixel, huấn luyện lại Logistic Regression và so sánh Accuracy cùng độ bền vững trước nhiễu (Bài 3).

## 11. Tóm tắt
- Hồi quy Logistic mở rộng sang đa lớp bằng **softmax** và Cross-Entropy.
- Với đa lớp: dùng ma trận nhầm lẫn 10×10, F1 **macro/weighted**, ROC-AUC OvR và **Log Loss**; Accuracy chỉ đáng tin khi các lớp cân bằng.
- So sánh mô hình phải dùng **cross-validation** và xét độ lệch chuẩn — chênh lệch nhỏ hơn nhiễu thì không kết luận được ai hơn.
- Mô hình đơn giản (Logistic) thường là **baseline mạnh**; chọn mô hình cuối theo độ chính xác, chi phí, khả năng giải thích và độ bền vững, không chỉ một con số.

➡️ Chương 7: thuật toán K-Láng giềng gần nhất (K-NN).